<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">SendGrid (optional)</h2>
            <span style="color:#ff7800;">Same as the main lab: you can use SendGrid or alternatives from the course FAQ (Q29).</span>
        </td>
    </tr>
</table>

In [ ]:

from pathlib import Path
import os

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict, List, Any
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
import csv
import json


def load_env_robust() -> None:
    candidates = {Path.cwd().resolve()}
    try:
        candidates.add(Path(__file__).resolve().parent)
    except NameError:
        pass
    for base in list(candidates):
        for p in [base, *base.parents]:
            env_path = p / ".env"
            if env_path.is_file():
                load_dotenv(env_path, override=True)
                return
    load_dotenv(override=True)


load_env_robust()

NOTEBOOK_DIR = Path.cwd()
for cand in [Path.cwd(), Path.cwd() / "2_openai/community_contributions/Igniters-Week2-Tunde_Wey"]:
    if (cand / "prospects_sample.csv").is_file():
        NOTEBOOK_DIR = cand.resolve()
        break

SENDGRID_FROM = os.environ.get("SENDGRID_FROM_EMAIL", "your-verified-sender@example.com")
SENDGRID_TO = os.environ.get("SENDGRID_TO_EMAIL", "recipient@example.com")


In [ ]:

def send_test_email() -> None:
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(SENDGRID_FROM) #create a .env file and include sender's mail
    to_email = To(SENDGRID_TO) #create a .env file and include receiver's mail
    content = Content("text/plain", "Igniters Week2 — test email from community 2_lab2 notebook.")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)


# send_test_email()  # uncomment after setting .env


## Step 1: Agent workflow

In [ ]:

instructions1 = """You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."""

instructions2 = """You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."""

instructions3 = """You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."""


In [ ]:

sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini",
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini",
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini",
)


In [ ]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


In [ ]:

message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


In [ ]:

sales_picker = Agent(
    name="sales_picker",
    instructions="""You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.""",
    model="gpt-4o-mini",
)


In [ ]:

message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Traces: https://platform.openai.com/traces

## Part 2: Tools (`@function_tool`) and agents-as-tools

In [ ]:

@function_tool
def send_email(body: str) -> Dict[str, str]:
    """Send one plain-text email with the given body."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(SENDGRID_FROM)
    to_email = To(SENDGRID_TO)
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}


In [ ]:

description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools_basic = [tool1, tool2, tool3, send_email]
tools_basic


## Sales Manager — model chooses tools (Anthropic-style *agent*)

In [ ]:

instructions_manager = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.

3. Use the send_email tool to send the best email (and only the best email) to the user.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

# The crucial line vs pure "workflow": giving the LLM tools so it decides call order.
sales_manager = Agent(
    name="Sales Manager",
    instructions=instructions_manager,
    tools=tools_basic,
    model="gpt-4o-mini",
)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)


In [ ]:

subject_instructions = """You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."""

html_instructions = """You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."""

subject_writer = Agent(
    name="Email subject writer",
    instructions=subject_instructions,
    model="gpt-4o-mini",
)
subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="Write a subject for a cold sales email",
)

html_converter = Agent(
    name="HTML email body converter",
    instructions=html_instructions,
    model="gpt-4o-mini",
)
html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convert a text email body to an HTML email body",
)


In [ ]:

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send HTML email with subject."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(SENDGRID_FROM)
    to_email = To(SENDGRID_TO)
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}


In [ ]:

emailer_tools = [subject_tool, html_tool, send_html_email]

instructions_emailer = """You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."""

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions_emailer,
    tools=emailer_tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it",
)


In [ ]:

tools_sales_only = [tool1, tool2, tool3]
handoffs = [emailer_agent]

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.

3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager_h = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools_sales_only,
    handoffs=handoffs,
    model="gpt-4o-mini",
)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager_h, message)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise — solutions</h2>
        </td>
    </tr>
</table>

# Question 1. The Agentic design patterns I can identify in this lab are:

Role-based agents – Three sales agents with different personas (professional / engaging / busy).

Parallel execution – asyncio.gather runs multiple agents at the same time.

Evaluator / selector – sales_picker chooses the best draft from several outputs.

Tool use – @function_tool for send_email / send_html_email so the model can trigger real actions.

Agent as a tool – some_agent.as_tool(...) so one agent can invoke another and get a result back.

Orchestrator (manager) agent – Sales Manager coordinates drafts and sending using instructions + tools.

Handoff – Sales Manager hands off to the Email Manager so another agent owns formatting + sending.

Tracing – with trace("...") to inspect runs in the dashboard.

# Question 2. The “one line”: workflow → “agent” (Anthropic-style)
Anthropic’s idea: a workflow is mostly your code deciding the steps; an agent is when the model decides what to do next (e.g. which tool to call), within rules you give it.

In this notebook, the turn happens when you give the Sales Manager tools=... and run Runner.run(sales_manager, message) so the model picks tools (draft agents, then send), instead of you hard-coding gather → build text → Runner.run(sales_picker, ...).

The line people usually single out is the Agent(..., tools=tools, ...) for the Sales Manager (the tools= part is the hinge). If your instructor wants literally one line, quote that constructor line including tools=tools.

# Question 3. More tools / agents (e.g. mail merge)
Ideas that fit the same pattern:

@function_tool: e.g. load_prospects(path) → returns a list of {email, name}.

@function_tool: e.g. send_bulk_plaintext(bodies_and_recipients) or loop inside one tool that 
calls SendGrid per row (with rate limits / errors handled in code).

Extra agent-as-tool: e.g. “Legal tone checker” or “shorten to 120 words” before send.

Keep sensitive logic (API keys, recipient lists) in your functions; expose narrow tools with clear docstrings

# Question 4. HARD CHALLENGE: reply webhook + ongoing SDR
Rough architecture:

Inbound email – SendGrid Inbound Parse sends parsed replies to your HTTPS endpoint (you host a small web app).

Your app stores the reply, maps it to the original thread (e.g. via a unique token in Reply-To or subject).

You call your Sales Manager / SDR agent with the new message and history, then send the reply via SendGrid again.

Alternatives: third-party inbox APIs, or provider-specific “reply” handling—same idea: webhook → your server → agent → send.

You’ll deal with verification, idempotency (webhooks can retry), and secrets—normal “vibe coding” production stuff.

## Extension: mail merge + tone-check agent-as-tool

In [ ]:

def _resolve_csv_path(path: str) -> Path:
    p = Path(path)
    if p.is_file():
        return p.resolve()
    here = NOTEBOOK_DIR / path
    if here.is_file():
        return here.resolve()
    root_guess = Path.cwd()
    for parent in [root_guess, *root_guess.parents]:
        cand = parent / path
        if cand.is_file():
            return cand.resolve()
    raise FileNotFoundError(f"CSV not found: {path}")


def _load_prospects_impl(path: str) -> str:
    csv_path = _resolve_csv_path(path)
    rows: List[Dict[str, str]] = []
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = (row.get("name") or "").strip()
            email = (row.get("email") or "").strip()
            if name and email:
                rows.append({"name": name, "email": email})
    return json.dumps(rows)


@function_tool
def load_prospects_from_csv(path: str) -> str:
    """Load prospects from CSV with columns name,email. Returns JSON array string."""
    return _load_prospects_impl(path)


@function_tool
def send_plaintext_mail_merge(subject: str, body_template: str, prospects_json: str) -> Dict[str, Any]:
    """Send one plain-text email per prospect. body_template may include {name}. prospects_json: JSON array of {name, email}."""
    prospects: List[Dict[str, str]] = json.loads(prospects_json)
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(SENDGRID_FROM)
    sent = 0
    errors: List[str] = []
    for row in prospects:
        body = body_template.format(name=row["name"])
        to_email = To(row["email"])
        content = Content("text/plain", body)
        mail = Mail(from_email, to_email, subject, content).get()
        try:
            sg.client.mail.send.post(request_body=mail)
            sent += 1
        except Exception as e:
            errors.append(f"{row.get('email')}: {e}")
    return {"sent": sent, "errors": errors}


tone_checker = Agent(
    name="Tone compliance checker",
    instructions="""You review a cold email for risky or spammy claims. \
Reply with a short bullet list: issues found, or say 'OK' if acceptable.""",
    model="gpt-4o-mini",
)
tone_tool = tone_checker.as_tool(
    tool_name="check_email_tone",
    tool_description="Review email body for compliance/tone issues before sending",
)


In [ ]:

# Demo: load sample CSV (prospects_sample.csv next to this notebook)
sample = _load_prospects_impl("prospects_sample.csv")
print(sample)

# To actually send mail-merge, use the agent with send_plaintext_mail_merge as a tool,
# or call SendGrid in a small test script — avoid blasting fake example.com addresses.


In [ ]:

extended_tools = [tool1, tool2, tool3, tone_tool, load_prospects_from_csv, send_plaintext_mail_merge, send_email]

instructions_extended = """
You are a Sales Manager at ComplAI.

When asked for a single best cold email and send it:
1. Use all three sales_agent tools for drafts, pick the best, optionally run check_email_tone on the winner.
2. Send exactly one email with send_email.

When asked to mail-merge to a list:
1. Use load_prospects_from_csv with the path provided (e.g. prospects_sample.csv).
2. Draft one template (you may use sales agents) then use send_plaintext_mail_merge with {name} in the body.
3. Only send if the user clearly asked to mail-merge; otherwise ask for confirmation.

Never send real mail-merge to example.com addresses in the sample CSV without replacing them.
"""

sales_manager_extended = Agent(
    name="Sales Manager (extended)",
    instructions=instructions_extended,
    tools=extended_tools,
    model="gpt-4o-mini",
)

# Example (costs tokens + may send email if CSV has real addresses):
# with trace("Extended SDR"):
#     await Runner.run(
#         sales_manager_extended,
#         "Write one cold email for Dear CEO, check tone, send with send_email only.",
#     )
